In [1]:
import pandas as pd
from pathlib import Path

INPUT_FILE = Path("CYTB_results/CYTB_all_candidates.csv")
OUTPUT_FILE = Path("CYTB_results/CYTB_selected_sequences.csv")

df = pd.read_csv(INPUT_FILE, sep=None, engine="python")

df["Partial"] = (
    df["Partial"]
    .astype(str)
    .str.lower()
    .isin(["true", "1", "yes"])
)

df["Length_difference"] = abs(df["Length_aa"] - 380)

# Предпочитаем полные последовательности
df = df.sort_values(
    by=["Partial", "Length_difference", "Score"],
    ascending=[True, True, False]
)

# Определяем столбец с видом
species_column = "Species" if "Species" in df.columns else "NCBI_organism"

# Выбираем одну последовательность на вид
selected = df.drop_duplicates(subset=species_column, keep="first")

selected.to_csv(OUTPUT_FILE, index=False)

print(f"Всего кандидатов: {len(df)}")
print(f"Выбрано последовательностей: {len(selected)}")
print(f"Видов: {selected[species_column].nunique()}")
print(f"Файл сохранён: {OUTPUT_FILE}")

print("\nВыбранные последовательности:")
print(selected[
    [species_column, "Accession", "Length_aa", "Partial", "Score"]
].to_string(index=False))

KeyError: 'Score'

In [9]:
import pandas as pd
from Bio import SeqIO
from pathlib import Path

BASE_DIR = Path("CYTB_results")

CSV_FILE = BASE_DIR / "CYTB_selected_sequences.csv"
FASTA_FILE = BASE_DIR / "CYTB_all_sequences.fasta"
OUTPUT_FASTA = BASE_DIR / "CYTB_selected_sequences.fasta"
MISSING_FILE = BASE_DIR / "CYTB_selected_missing_accessions.txt"

df = pd.read_csv(CSV_FILE, encoding="utf-8-sig")

selected = {
    str(row["Accession"]).strip(): str(row["Host_species"]).strip()
    for _, row in df.iterrows()
}

selected_records = {}
for record in SeqIO.parse(FASTA_FILE, "fasta"):
    if record.id in selected:
        selected_records[record.id] = record

output_records = []
for accession, species in selected.items():
    if accession in selected_records:
        record = selected_records[accession]
        record.id = species.replace(" ", "_")
        record.name = record.id
        record.description = ""
        output_records.append(record)

SeqIO.write(output_records, OUTPUT_FASTA, "fasta")

missing = set(selected) - set(selected_records)

with open(MISSING_FILE, "w", encoding="utf-8") as f:
    f.write("\n".join(sorted(missing)))

print(f"Выбрано в CSV: {len(selected)}")
print(f"Найдено в FASTA: {len(output_records)}")
print(f"Не найдено: {len(missing)}")
print(f"FASTA: {OUTPUT_FASTA}")

Выбрано в CSV: 28
Найдено в FASTA: 28
Не найдено: 0
FASTA: CYTB_results/CYTB_selected_sequences.fasta


In [7]:
from Bio import SeqIO

for record in SeqIO.parse("CYTB_results/CYTB_all_sequences.fasta", "fasta"):
    print(record.description)
    if record:
        break

YHG28083.1 cytochrome b (mitochondrion) [Gallus gallus]
